# Uno Coefficient Relationships

This notebook uses the current probe-simulation and Uno phase conventions to test one aberration coefficient family at a time. It computes the Uno et al. profile coefficients from paired `C1_offset=-909 nm` and `C1_offset=+909 nm` probes, then plots the relationship between the input aberration coefficient and the recovered scalar values `C1_value`/`C3_value` plus the harmonic values `A1_value`, `B2_value`, `A2_value`, `A3_value`, and `S3_value`.

The scalar diagnostic definitions used here are `C1_value = -Cdf_value = -(1/N) sum_k (Xigma_under,k - Xigma_over,k)` and `C3_value = -mean(Rho_over)`. The original Uno-style `Cdf_value` and paired-focus `Cs_value = (1/N) sum_k (Rho_under,k - Rho_over,k)` are still retained in the CSV for comparison.

C1 and C3 are scalar sweeps, so they are plotted as `C1_value` vs `C1` and overfocus-only `C3_value` vs `C3`. A1, B2/C21, A2, A3, and S3/C32 are harmonic sweeps, so they include amplitude and phase plots. The notebook also saves Fig. 8-style probe-shape galleries for representative values from every sweep, with color bars on the probe images. A separate C1/C3 grid sweep maps `C1_value` and `C3_value` over coupled `(C1, C3)` input combinations. The convention source of truth is `src/aberration_simulation/uno_conventions.py`. The notebook imports `UNO_HARMONIC_ORDERS`, `PRIMARY_PHASE_CONVENTIONS`, and `add_complex_columns()` from that module instead of redefining phase conventions locally.


## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone or Pull Latest GitHub Code

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
print("repo root:", ROOT)
print("commit:", commit)

## 3. Install and Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")

print("Dependencies are ready.")

## 4. Imports and Current Conventions

In [ ]:
from pathlib import Path
import csv
import zipfile

import matplotlib.pyplot as plt
import numpy as np

from aberration_simulation.backend import HAS_CUPY, asnumpy, xp
from aberration_simulation.line_profiles import extract_line_profiles_from_stack
from aberration_simulation.optics import SimulationConfig, run_simulation, save_npz
from aberration_simulation.uno_conventions import (
    PRIMARY_PHASE_CONVENTIONS,
    UNO_HARMONIC_ORDERS,
    add_complex_columns,
)

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())
print("Uno primary phase conventions:")
for name, convention in PRIMARY_PHASE_CONVENTIONS.items():
    period = 360.0 / UNO_HARMONIC_ORDERS[name]
    print(f"  {name}: sign={convention['sign']:g}, offset={convention['offset_deg']:g} deg, period={period:g} deg")

## 5. Build One-Coefficient-at-a-Time Sweep

In [ ]:
OUTPUT_DIR = ROOT / "outputs" / "uno_relationships"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNDER_FOCUS_C1_OFFSET = -909
OVER_FOCUS_C1_OFFSET = 909
C1_OFFSETS = [UNDER_FOCUS_C1_OFFSET, OVER_FOCUS_C1_OFFSET]
PROFILE_RADIUS_PIXELS = 80
PROFILE_STEP_DEGREES = 10
NUM_PROFILE_LINES = int(180 / PROFILE_STEP_DEGREES) + 1

C1_C3_GRID_LABEL = "C1_C3_grid"
C1_C3_GRID_C1_VALUES = [-80, -40, -20, 0, 20, 40, 80]
C1_C3_GRID_C3_VALUES = [0, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0]

A1_S3_GRID_LABEL = "A1_S3_grid"
A1_S3_GRID_A1_AMP_VALUES = [0, 30, 60]
A1_S3_GRID_S3_AMP_VALUES = [0, 8, 16]
A1_S3_GRID_A1_PHASE_VALUES = [0, 45, 90, 135]
A1_S3_GRID_S3_PHASE_VALUES = [0, 45, 90, 135]

BASELINE_PARAMETERS = {
    "C1_offset": 0,
    "A3_amp": 0,
    "S3_amp": 0,
    "A2_amp": 0,
    "B2_amp": 0,
    "C1": 0,
    "C3": 0,
    "A1_amp": 0,
    "A1_phase": 0,
    "A2_phase": 0,
    "A3_phase": 0,
    "S3_phase": 0,
    "B2_phase": 0,
}

SCALAR_SWEEP_SPECS = [
    {
        "label": "C1",
        "value_name": "C1_value",
        "input_field": "C1",
        "input_values": [-80, -40, -20, 20, 40, 80],
    },
    {
        "label": "C3",
        "value_name": "C3_value",
        "input_field": "C3",
        "input_values": [0.1, 0.2, 0.5, 1.0, 1.5, 2.0],
    },
]

HARMONIC_SWEEP_SPECS = [
    {
        "label": "A1",
        "value_name": "A1_value",
        "amp_field": "A1_amp",
        "phase_field": "A1_phase",
        "amp_values": [5, 15, 30, 60],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
    {
        "label": "B2/C21",
        "value_name": "B2_value",
        "amp_field": "B2_amp",
        "phase_field": "B2_phase",
        "amp_values": [0.5, 1.0, 2.0, 3.0],
        "phase_values": [0, 45, 90, 135, 180, 225, 270, 315],
    },
    {
        "label": "A2",
        "value_name": "A2_value",
        "amp_field": "A2_amp",
        "phase_field": "A2_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 30, 60, 90, 120],
    },
    {
        "label": "A3",
        "value_name": "A3_value",
        "amp_field": "A3_amp",
        "phase_field": "A3_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
    {
        "label": "S3/C32",
        "value_name": "S3_value",
        "amp_field": "S3_amp",
        "phase_field": "S3_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
]

SWEEP_SPECS = SCALAR_SWEEP_SPECS + HARMONIC_SWEEP_SPECS

COMBINATION_FIELDS = (
    "sweep_label",
    "C1",
    "A1_amp",
    "A1_phase",
    "A2_amp",
    "A2_phase",
    "B2_amp",
    "B2_phase",
    "A3_amp",
    "A3_phase",
    "S3_amp",
    "S3_phase",
    "C3",
)

base_cases = []
for spec in SCALAR_SWEEP_SPECS:
    for value in spec["input_values"]:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = spec["label"]
        params[spec["input_field"]] = value
        base_cases.append(params)

for spec in HARMONIC_SWEEP_SPECS:
    for amp in spec["amp_values"]:
        for phase in spec["phase_values"]:
            params = dict(BASELINE_PARAMETERS)
            params["sweep_label"] = spec["label"]
            params[spec["amp_field"]] = amp
            params[spec["phase_field"]] = phase
            base_cases.append(params)

for c1_value in C1_C3_GRID_C1_VALUES:
    for c3_value in C1_C3_GRID_C3_VALUES:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = C1_C3_GRID_LABEL
        params["C1"] = c1_value
        params["C3"] = c3_value
        base_cases.append(params)

for a1_amp in A1_S3_GRID_A1_AMP_VALUES:
    for s3_amp in A1_S3_GRID_S3_AMP_VALUES:
        for a1_phase in A1_S3_GRID_A1_PHASE_VALUES:
            for s3_phase in A1_S3_GRID_S3_PHASE_VALUES:
                params = dict(BASELINE_PARAMETERS)
                params["sweep_label"] = A1_S3_GRID_LABEL
                params["A1_amp"] = a1_amp
                params["A1_phase"] = a1_phase
                params["S3_amp"] = s3_amp
                params["S3_phase"] = s3_phase
                base_cases.append(params)

parameters = []
for base_case in base_cases:
    for c1_offset in C1_OFFSETS:
        params = dict(base_case)
        params["C1_offset"] = c1_offset
        parameters.append(params)

print("base cases, including coupled grids:", len(base_cases))
print("simulated probe images, including C1 offsets:", len(parameters))
for spec in SWEEP_SPECS:
    count = sum(1 for case in base_cases if case["sweep_label"] == spec["label"])
    print(f"  {spec['label']}: {count} base cases")

## 6. Run Probe Simulation

In [ ]:
config = SimulationConfig(
    pix_dim=(256, 256),
    real_dim=(1280, 1280),
    app=30,
    sigma=2,
)

probe_images, selected = run_simulation(config, parameters)
# `selected` is rebuilt by the simulation backend from numeric fields only.
# Keep the original records for sweep labels and under/over pairing metadata.
simulation_records = [dict(params) for params in parameters]
probe_np = asnumpy(probe_images)
print("probe image stack:", probe_np.shape)
print("metadata records:", len(simulation_records))
print("intensity range:", float(np.min(probe_np)), float(np.max(probe_np)))
print("per-image intensity sums, min/max:", float(np.min(np.sum(probe_np, axis=(0, 1)))), float(np.max(np.sum(probe_np, axis=(0, 1)))))

NPZ_PATH = OUTPUT_DIR / "uno_relationship_probe_images.npz"
save_npz(NPZ_PATH, probe_images, selected)
print("saved:", NPZ_PATH)

## 7. Uno Formula Implementation

The profile quantities are evaluated on line-profile angles `theta_k` with `N` sampled angles. The notebook stores the original paired-focus scalar terms and the sign-adjusted scalar diagnostics used in the relationship plots:

- `Cdf_value = (1/N) sum_k (Xigma_under,k - Xigma_over,k)`
- `C1_value = -Cdf_value`
- `A1_value = (2/N) sum_k (Xigma_under,k - Xigma_over,k) exp(2 i theta_k)`
- `B2_value = (2/N) sum_k (Mu_under,k + Mu_over,k) exp(i theta_k)`
- `A2_value = (2/N) sum_k (Mu_under,k + Mu_over,k) exp(3 i theta_k)`
- `Cs_value = (1/N) sum_k (Rho_under,k - Rho_over,k)`
- `C3_value = -mean(Rho_over) = -(1/N) sum_k Rho_over,k`
- `S3_value = (2/N) sum_k (Rho_under,k - Rho_over,k) exp(2 i theta_k)`
- `A3_value = (2/N) sum_k (Xigma_under,k - Xigma_over,k) exp(4 i theta_k)`


In [ ]:
def compute_line_characteristics(profiles_np, radius):
    """Compute Xigma, Mu, and Rho for each angular line profile."""
    j = np.arange(-radius, radius + 1, dtype=float)
    center_index = int(np.argmin(np.abs(j)))
    p0 = profiles_np[:, center_index]

    W = np.sum(profiles_np, axis=1)
    T = np.sum(profiles_np ** 2, axis=1)
    W = np.where(W == 0, np.nan, W)
    T = np.where(T == 0, np.nan, T)

    Xigma = np.sqrt(np.sum((j[None, :] ** 2) * profiles_np, axis=1) / W)
    Mu = np.sum(j[None, :] * profiles_np, axis=1) / W

    nonzero = j != 0
    curvature_sum = np.sum(
        ((profiles_np[:, nonzero] - p0[:, None]) * profiles_np[:, nonzero])
        / np.abs(j[nonzero])[None, :],
        axis=1,
    )
    Rho = (Xigma ** 2 / T) * curvature_sum

    return {"Xigma": Xigma, "Mu": Mu, "Rho": Rho}


def compute_uno_values(under_chars, over_chars, angles_deg):
    theta = np.deg2rad(angles_deg)
    N = len(theta)

    Xigma_diff = under_chars["Xigma"] - over_chars["Xigma"]
    Mu_sum = under_chars["Mu"] + over_chars["Mu"]
    Rho_diff = under_chars["Rho"] - over_chars["Rho"]

    Cdf_value = np.sum(Xigma_diff) / N
    C1_value = -Cdf_value
    A1_value = 2 * np.sum(Xigma_diff * np.exp(2j * theta)) / N
    B2_value = 2 * np.sum(Mu_sum * np.exp(1j * theta)) / N
    A2_value = 2 * np.sum(Mu_sum * np.exp(3j * theta)) / N
    Cs_value = np.sum(Rho_diff) / N
    C3_value = -np.sum(over_chars["Rho"]) / N
    S3_value = 2 * np.sum(Rho_diff * np.exp(2j * theta)) / N
    A3_value = 2 * np.sum(Xigma_diff * np.exp(4j * theta)) / N

    return {
        "Cdf_value": Cdf_value,
        "C1_value": C1_value,
        "A1_value": A1_value,
        "B2_value": B2_value,
        "A2_value": A2_value,
        "Cs_value": Cs_value,
        "C3_value": C3_value,
        "S3_value": S3_value,
        "A3_value": A3_value,
    }


def combination_key(params):
    return tuple(params.get(field, 0.0) for field in COMBINATION_FIELDS)


def select_under_over_pairs(parameters):
    pairs = {}
    representatives = {}
    for index, params in enumerate(parameters):
        key = combination_key(params)
        representatives.setdefault(key, params)
        pair = pairs.setdefault(key, {})
        if np.isclose(params["C1_offset"], UNDER_FOCUS_C1_OFFSET):
            pair["under"] = index
        if np.isclose(params["C1_offset"], OVER_FOCUS_C1_OFFSET):
            pair["over"] = index

    selected_pairs = []
    for key, pair in pairs.items():
        if "under" in pair and "over" in pair:
            selected_pairs.append((representatives[key], pair["under"], pair["over"]))
    return selected_pairs

## 8. Compute Uno Coefficients From Line Profiles

In [ ]:
pairs = select_under_over_pairs(simulation_records)
rows = []
print("under/over pairs:", len(pairs))

for case_index, (params, under_index, over_index) in enumerate(pairs):
    stack = probe_images[:, :, [under_index, over_index]]
    profiles, coords = extract_line_profiles_from_stack(
        stack,
        num_lines=NUM_PROFILE_LINES,
        radius=PROFILE_RADIUS_PIXELS,
    )
    profiles_np = asnumpy(profiles)
    angles_deg = np.asarray(coords["angles_deg"], dtype=float)

    under_chars = compute_line_characteristics(profiles_np[:, :, 0], PROFILE_RADIUS_PIXELS)
    over_chars = compute_line_characteristics(profiles_np[:, :, 1], PROFILE_RADIUS_PIXELS)
    uno_values = compute_uno_values(under_chars, over_chars, angles_deg)

    row = {field: params.get(field, 0.0) for field in COMBINATION_FIELDS}
    row["case_index"] = case_index
    row["under_index"] = under_index
    row["over_index"] = over_index
    row["under_C1_offset"] = UNDER_FOCUS_C1_OFFSET
    row["over_C1_offset"] = OVER_FOCUS_C1_OFFSET
    for name, value in uno_values.items():
        add_complex_columns(row, name, value)
    rows.append(row)

CSV_PATH = OUTPUT_DIR / "uno_coefficient_relationships.csv"
with CSV_PATH.open("w", newline="") as handle:
    fieldnames = list(rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("computed paired cases:", len(rows))
row_counts = {spec["label"]: sum(row["sweep_label"] == spec["label"] for row in rows) for spec in SWEEP_SPECS}
row_counts[C1_C3_GRID_LABEL] = sum(row["sweep_label"] == C1_C3_GRID_LABEL for row in rows)
row_counts[A1_S3_GRID_LABEL] = sum(row["sweep_label"] == A1_S3_GRID_LABEL for row in rows)
print("rows by sweep:", row_counts)
missing_sweeps = [label for label, count in row_counts.items() if count == 0]
if missing_sweeps:
    raise ValueError(f"No computed rows for sweeps: {missing_sweeps}")
print("saved:", CSV_PATH)
print("first row:")
for key in ["sweep_label", "A1_value_abs", "B2_value_abs", "A2_value_abs", "A3_value_abs", "S3_value_abs"]:
    print(" ", key, rows[0].get(key))

## 9. Relationship Plots

In [ ]:
NOTEBOOK_FIX_VERSION = "metadata-preserved-c1-c3"
print("relationship notebook plotting cell version:", NOTEBOOK_FIX_VERSION)

def circular_difference_deg(a, b, period):
    return np.abs((a - b + period / 2) % period - period / 2)


def fitted_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y) & (x != 0)
    if not np.any(valid):
        return np.nan
    return float(np.sum(x[valid] * y[valid]) / np.sum(x[valid] ** 2))


def plot_scalar_relationship_for_spec(spec):
    value_name = spec["value_name"]
    input_field = spec["input_field"]
    selected_rows = [row for row in rows if row["sweep_label"] == spec["label"]]
    if not selected_rows:
        raise ValueError(f"No rows found for {spec['label']}. Check sweep_label metadata.")

    input_value = np.asarray([row[input_field] for row in selected_rows], dtype=float)
    output_value = np.asarray([row[value_name + "_real"] for row in selected_rows], dtype=float)
    slope = fitted_slope(input_value, output_value)
    residual = output_value - slope * input_value if np.isfinite(slope) else np.full_like(output_value, np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    axes[0].scatter(input_value, output_value, s=48, alpha=0.9)
    x_min = float(np.nanmin(input_value))
    x_max = float(np.nanmax(input_value))
    x_line = np.linspace(x_min, x_max, 100)
    if np.isfinite(slope):
        axes[0].plot(x_line, slope * x_line, color="black", linestyle="--", linewidth=1, label=f"fit slope={slope:.4g}")
        axes[0].legend(fontsize=8)
    axes[0].set_title(f"{value_name} vs {input_field}")
    axes[0].set_xlabel(input_field)
    axes[0].set_ylabel(value_name)
    axes[0].grid(alpha=0.3)

    axes[1].scatter(input_value, residual, s=48, alpha=0.9)
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[1].set_title("linear-fit residual")
    axes[1].set_xlabel(input_field)
    axes[1].set_ylabel(f"{value_name} residual")
    axes[1].grid(alpha=0.3)

    fig.suptitle(f"{spec['label']}: scalar one-coefficient sweep", fontsize=12)
    fig.tight_layout()
    plot_path = OUTPUT_DIR / f"relationship_{value_name}.png"
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()

    print(
        spec["label"],
        "cases=", len(selected_rows),
        "slope=", slope,
        "mean abs residual=", float(np.nanmean(np.abs(residual))),
        "plot=", plot_path,
    )
    return plot_path


def plot_harmonic_relationship_for_spec(spec):
    value_name = spec["value_name"]
    amp_field = spec["amp_field"]
    phase_field = spec["phase_field"]
    order = UNO_HARMONIC_ORDERS[value_name]
    period = 360.0 / order
    selected_rows = [row for row in rows if row["sweep_label"] == spec["label"]]
    if not selected_rows:
        raise ValueError(f"No rows found for {spec['label']}. Check sweep_label metadata.")

    input_amp = np.asarray([row[amp_field] for row in selected_rows], dtype=float)
    input_phase = np.asarray([row[phase_field] % period for row in selected_rows], dtype=float)
    output_amp = np.asarray([row[value_name + "_abs"] for row in selected_rows], dtype=float)
    output_phase = np.asarray([row[value_name + "_phase_deg"] for row in selected_rows], dtype=float)
    phase_error = circular_difference_deg(output_phase, input_phase, period)
    slope = fitted_slope(input_amp, output_amp)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    amp_scatter = axes[0].scatter(input_amp, output_amp, c=input_phase, cmap="twilight", s=42, alpha=0.9)
    x_max = float(np.nanmax(input_amp))
    x_line = np.linspace(0, x_max * 1.05, 100)
    if np.isfinite(slope):
        axes[0].plot(x_line, slope * x_line, color="black", linestyle="--", linewidth=1, label=f"fit slope={slope:.4g}")
        axes[0].legend(fontsize=8)
    axes[0].set_title(f"{value_name} amplitude vs {amp_field}")
    axes[0].set_xlabel(amp_field)
    axes[0].set_ylabel(f"abs({value_name})")
    axes[0].grid(alpha=0.3)
    fig.colorbar(amp_scatter, ax=axes[0], label=f"{phase_field} mod {period:g} deg")

    phase_scatter = axes[1].scatter(input_phase, output_phase, c=input_amp, cmap="viridis", s=42, alpha=0.9)
    axes[1].plot([0, period], [0, period], color="black", linestyle="--", linewidth=1)
    axes[1].set_xlim(-0.04 * period, 1.04 * period)
    axes[1].set_ylim(-0.04 * period, 1.04 * period)
    axes[1].set_title(f"{value_name} phase vs {phase_field}")
    axes[1].set_xlabel(f"{phase_field} mod period (deg)")
    axes[1].set_ylabel(f"{value_name}_phase_deg")
    axes[1].grid(alpha=0.3)
    fig.colorbar(phase_scatter, ax=axes[1], label=amp_field)

    axes[2].scatter(input_amp, phase_error, c=input_phase, cmap="twilight", s=42, alpha=0.9)
    axes[2].set_title("phase error")
    axes[2].set_xlabel(amp_field)
    axes[2].set_ylabel("abs wrapped phase error (deg)")
    axes[2].grid(alpha=0.3)

    fig.suptitle(
        f"{spec['label']}: one-coefficient sweep | convention {PRIMARY_PHASE_CONVENTIONS[value_name]}",
        fontsize=12,
    )
    fig.tight_layout()
    plot_path = OUTPUT_DIR / f"relationship_{value_name}.png"
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()

    print(
        spec["label"],
        "cases=", len(selected_rows),
        "slope=", slope,
        "mean phase error=", float(np.nanmean(phase_error)),
        "max phase error=", float(np.nanmax(phase_error)),
        "plot=", plot_path,
    )
    return plot_path


plot_paths = []
plot_paths.extend(plot_scalar_relationship_for_spec(spec) for spec in SCALAR_SWEEP_SPECS)
plot_paths.extend(plot_harmonic_relationship_for_spec(spec) for spec in HARMONIC_SWEEP_SPECS)


In [ ]:
def plot_c1_c3_coupling_maps():
    selected_rows = [row for row in rows if row["sweep_label"] == C1_C3_GRID_LABEL]
    if not selected_rows:
        raise ValueError(f"No rows found for {C1_C3_GRID_LABEL}.")

    c1_values = np.asarray(sorted({float(row["C1"]) for row in selected_rows}), dtype=float)
    c3_values = np.asarray(sorted({float(row["C3"]) for row in selected_rows}), dtype=float)
    c1_index = {value: index for index, value in enumerate(c1_values)}
    c3_index = {value: index for index, value in enumerate(c3_values)}

    c1_map = np.full((len(c3_values), len(c1_values)), np.nan, dtype=float)
    c3_map = np.full_like(c1_map, np.nan)
    for row in selected_rows:
        iy = c3_index[float(row["C3"])]
        ix = c1_index[float(row["C1"])]
        c1_map[iy, ix] = float(row["C1_value_real"])
        c3_map[iy, ix] = float(row["C3_value_real"])

    extent = [
        float(c1_values.min()),
        float(c1_values.max()),
        float(c3_values.min()),
        float(c3_values.max()),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), squeeze=False)
    for axis, data, title, label in [
        (axes[0, 0], c1_map, "C1_value over coupled C1/C3 sweep", "C1_value"),
        (axes[0, 1], c3_map, "C3_value over coupled C1/C3 sweep", "C3_value"),
    ]:
        im = axis.imshow(data, origin="lower", aspect="auto", extent=extent, cmap="viridis")
        axis.set_title(title)
        axis.set_xlabel("input C1")
        axis.set_ylabel("input C3")
        axis.set_xticks(c1_values)
        axis.set_yticks(c3_values)
        axis.grid(color="white", alpha=0.25, linewidth=0.7)
        fig.colorbar(im, ax=axis, label=label)

    fig.suptitle("Coupled scalar response maps for C1 and C3", fontsize=12)
    fig.tight_layout()
    plot_path = OUTPUT_DIR / "relationship_c1_c3_coupling_maps.png"
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", plot_path)
    return plot_path


coupling_map_path = plot_c1_c3_coupling_maps()
plot_paths.append(coupling_map_path)


In [ ]:
def plot_a1_s3_coupling_map(value_name, value_column, output_name):
    selected_rows = [row for row in rows if row["sweep_label"] == A1_S3_GRID_LABEL]
    if not selected_rows:
        raise ValueError(f"No rows found for {A1_S3_GRID_LABEL}.")

    a1_amps = np.asarray(sorted({float(row["A1_amp"]) for row in selected_rows}), dtype=float)
    s3_amps = np.asarray(sorted({float(row["S3_amp"]) for row in selected_rows}), dtype=float)
    a1_phases = np.asarray(sorted({float(row["A1_phase"]) for row in selected_rows}), dtype=float)
    s3_phases = np.asarray(sorted({float(row["S3_phase"]) for row in selected_rows}), dtype=float)

    phase_maps = {}
    values_for_limits = []
    for a1_phase in a1_phases:
        for s3_phase in s3_phases:
            data = np.full((len(s3_amps), len(a1_amps)), np.nan, dtype=float)
            for row in selected_rows:
                if not np.isclose(float(row["A1_phase"]), a1_phase):
                    continue
                if not np.isclose(float(row["S3_phase"]), s3_phase):
                    continue
                iy = int(np.where(np.isclose(s3_amps, float(row["S3_amp"])))[0][0])
                ix = int(np.where(np.isclose(a1_amps, float(row["A1_amp"])))[0][0])
                data[iy, ix] = float(row[value_column])
            phase_maps[(a1_phase, s3_phase)] = data
            values_for_limits.append(data[np.isfinite(data)])

    finite_values = np.concatenate([values for values in values_for_limits if values.size])
    vmin = float(np.nanmin(finite_values))
    vmax = float(np.nanmax(finite_values))
    extent = [float(a1_amps.min()), float(a1_amps.max()), float(s3_amps.min()), float(s3_amps.max())]

    fig, axes = plt.subplots(
        len(s3_phases),
        len(a1_phases),
        figsize=(3.0 * len(a1_phases), 2.7 * len(s3_phases)),
        squeeze=False,
        sharex=True,
        sharey=True,
    )
    last_image = None
    for row_index, s3_phase in enumerate(s3_phases):
        for col_index, a1_phase in enumerate(a1_phases):
            axis = axes[row_index, col_index]
            data = phase_maps[(a1_phase, s3_phase)]
            last_image = axis.imshow(
                data,
                origin="lower",
                aspect="auto",
                extent=extent,
                cmap="viridis",
                vmin=vmin,
                vmax=vmax,
            )
            axis.set_title(f"A1 phase={a1_phase:g} deg\nS3 phase={s3_phase:g} deg", fontsize=8)
            axis.set_xticks(a1_amps)
            axis.set_yticks(s3_amps)
            axis.grid(color="white", alpha=0.22, linewidth=0.6)
            if row_index == len(s3_phases) - 1:
                axis.set_xlabel("A1_amp")
            if col_index == 0:
                axis.set_ylabel("S3_amp")

    fig.suptitle(f"{value_name} over coupled A1/S3 sweep | color = {value_column}", fontsize=12)
    fig.subplots_adjust(top=0.88, right=0.88, hspace=0.35, wspace=0.18)
    colorbar_axis = fig.add_axes([0.91, 0.14, 0.018, 0.70])
    fig.colorbar(last_image, cax=colorbar_axis, label=value_column)
    plot_path = OUTPUT_DIR / output_name
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", plot_path)
    return plot_path


a1_s3_coupling_paths = [
    plot_a1_s3_coupling_map("A1_value", "A1_value_abs", "relationship_a1_s3_coupling_a1_value_abs.png"),
    plot_a1_s3_coupling_map("S3_value", "S3_value_abs", "relationship_a1_s3_coupling_s3_value_abs.png"),
]
plot_paths.extend(a1_s3_coupling_paths)


In [ ]:
def row_float(row, name):
    return float(row[name])


def complex_coefficient_vector(row, prefix, order):
    amp = row_float(row, f"{prefix}_amp")
    phase = np.deg2rad(order * row_float(row, f"{prefix}_phase"))
    return np.asarray([amp * np.cos(phase), amp * np.sin(phase)], dtype=float)


def unique_rows_by_input(selected_rows, input_vector_fn, decimals=8):
    unique_rows = []
    seen = set()
    for row in selected_rows:
        key = tuple(np.round(input_vector_fn(row), decimals))
        if key in seen:
            continue
        seen.add(key)
        unique_rows.append(row)
    return unique_rows


def pairwise_uniqueness_metrics(selected_rows, input_vector_fn, output_vector_fn):
    input_vectors = np.asarray([input_vector_fn(row) for row in selected_rows], dtype=float)
    output_vectors = np.asarray([output_vector_fn(row) for row in selected_rows], dtype=float)
    pair_input_distances = []
    pair_output_distances = []
    nearest_output_distance = np.full(len(selected_rows), np.nan, dtype=float)
    nearest_input_distance = np.full(len(selected_rows), np.nan, dtype=float)
    nearest_index = np.full(len(selected_rows), -1, dtype=int)

    for i in range(len(selected_rows)):
        best_distance = np.inf
        for j in range(len(selected_rows)):
            if i == j:
                continue
            input_distance = float(np.linalg.norm(input_vectors[i] - input_vectors[j]))
            output_distance = float(np.linalg.norm(output_vectors[i] - output_vectors[j]))
            if j > i:
                pair_input_distances.append(input_distance)
                pair_output_distances.append(output_distance)
            if output_distance < best_distance:
                best_distance = output_distance
                nearest_output_distance[i] = output_distance
                nearest_input_distance[i] = input_distance
                nearest_index[i] = j

    return {
        "input_vectors": input_vectors,
        "output_vectors": output_vectors,
        "pair_input_distances": np.asarray(pair_input_distances, dtype=float),
        "pair_output_distances": np.asarray(pair_output_distances, dtype=float),
        "nearest_output_distance": nearest_output_distance,
        "nearest_input_distance": nearest_input_distance,
        "nearest_index": nearest_index,
    }


def plot_coupling_uniqueness_diagnostics(
    label,
    input_vector_fn,
    output_vector_fn,
    nearest_map_fn,
    output_name,
):
    selected_rows = [row for row in rows if row["sweep_label"] == label]
    if not selected_rows:
        raise ValueError(f"No rows found for {label}.")
    unique_rows = unique_rows_by_input(selected_rows, input_vector_fn)
    metrics = pairwise_uniqueness_metrics(unique_rows, input_vector_fn, output_vector_fn)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), squeeze=False)
    axes[0, 0].scatter(
        metrics["pair_input_distances"],
        metrics["pair_output_distances"],
        s=12,
        alpha=0.35,
    )
    min_pair_index = int(np.nanargmin(metrics["pair_output_distances"]))
    axes[0, 0].scatter(
        metrics["pair_input_distances"][min_pair_index],
        metrics["pair_output_distances"][min_pair_index],
        color="red",
        s=42,
        label="nearest output pair",
    )
    axes[0, 0].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[0, 0].set_title("pairwise separation")
    axes[0, 0].set_xlabel("input-space distance")
    axes[0, 0].set_ylabel("output-space distance")
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].legend(fontsize=8)

    nearest_map_fn(axes[0, 1], unique_rows, metrics)
    fig.suptitle(
        f"Uniqueness diagnostic: {label} | unique inputs={len(unique_rows)}",
        fontsize=12,
    )
    fig.tight_layout()
    plot_path = OUTPUT_DIR / output_name
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()

    min_output_index = int(np.nanargmin(metrics["nearest_output_distance"]))
    neighbor_index = int(metrics["nearest_index"][min_output_index])
    print("saved:", plot_path)
    print(
        label,
        "nearest output distance=", float(metrics["nearest_output_distance"][min_output_index]),
        "input distance for that pair=", float(metrics["nearest_input_distance"][min_output_index]),
    )
    print("  row:", unique_rows[min_output_index])
    print("  nearest row:", unique_rows[neighbor_index])
    return plot_path


def c1_c3_input_vector(row):
    return np.asarray([row_float(row, "C1"), row_float(row, "C3")], dtype=float)


def c1_c3_output_vector(row):
    return np.asarray([row_float(row, "C1_value_real"), row_float(row, "C3_value_real")], dtype=float)


def draw_c1_c3_nearest_map(axis, unique_rows, metrics):
    c1_values = np.asarray(sorted({row_float(row, "C1") for row in unique_rows}), dtype=float)
    c3_values = np.asarray(sorted({row_float(row, "C3") for row in unique_rows}), dtype=float)
    data = np.full((len(c3_values), len(c1_values)), np.nan, dtype=float)
    for row, distance in zip(unique_rows, metrics["nearest_output_distance"]):
        iy = int(np.where(np.isclose(c3_values, row_float(row, "C3")))[0][0])
        ix = int(np.where(np.isclose(c1_values, row_float(row, "C1")))[0][0])
        data[iy, ix] = distance
    image = axis.imshow(
        data,
        origin="lower",
        aspect="auto",
        extent=[float(c1_values.min()), float(c1_values.max()), float(c3_values.min()), float(c3_values.max())],
        cmap="magma",
    )
    axis.set_title("nearest output distance")
    axis.set_xlabel("input C1")
    axis.set_ylabel("input C3")
    axis.set_xticks(c1_values)
    axis.set_yticks(c3_values)
    axis.grid(color="white", alpha=0.25, linewidth=0.7)
    axis.figure.colorbar(image, ax=axis, label="nearest output distance")


def a1_s3_input_vector(row):
    return np.concatenate([
        complex_coefficient_vector(row, "A1", order=2),
        complex_coefficient_vector(row, "S3", order=2),
    ])


def a1_s3_output_vector(row):
    return np.asarray([
        row_float(row, "A1_value_real"),
        row_float(row, "A1_value_imag"),
        row_float(row, "S3_value_real"),
        row_float(row, "S3_value_imag"),
    ], dtype=float)


def draw_a1_s3_nearest_summary(axis, unique_rows, metrics):
    total_amp = np.asarray([
        np.hypot(row_float(row, "A1_amp"), row_float(row, "S3_amp"))
        for row in unique_rows
    ], dtype=float)
    scatter = axis.scatter(
        metrics["nearest_input_distance"],
        metrics["nearest_output_distance"],
        c=total_amp,
        cmap="viridis",
        s=46,
        alpha=0.85,
    )
    axis.axhline(0, color="black", linestyle="--", linewidth=1)
    axis.set_title("nearest output neighbor per input")
    axis.set_xlabel("input distance to nearest output neighbor")
    axis.set_ylabel("nearest output distance")
    axis.grid(alpha=0.3)
    axis.figure.colorbar(scatter, ax=axis, label="sqrt(A1_amp^2 + S3_amp^2)")


c1_c3_uniqueness_path = plot_coupling_uniqueness_diagnostics(
    C1_C3_GRID_LABEL,
    c1_c3_input_vector,
    c1_c3_output_vector,
    draw_c1_c3_nearest_map,
    "uniqueness_c1_c3.png",
)

a1_s3_uniqueness_path = plot_coupling_uniqueness_diagnostics(
    A1_S3_GRID_LABEL,
    a1_s3_input_vector,
    a1_s3_output_vector,
    draw_a1_s3_nearest_summary,
    "uniqueness_a1_s3_complex.png",
)
plot_paths.extend([c1_c3_uniqueness_path, a1_s3_uniqueness_path])


## 10. Summary Grid and Download Bundle

In [ ]:
summary_specs = SWEEP_SPECS
fig, axes = plt.subplots(2, len(summary_specs), figsize=(4.0 * len(summary_specs), 7.2), squeeze=False)

for column, spec in enumerate(summary_specs):
    selected_rows = [row for row in rows if row["sweep_label"] == spec["label"]]
    if not selected_rows:
        axes[0, column].set_title(spec["label"] + " (no rows)")
        axes[1, column].set_title(spec["label"] + " (no rows)")
        continue

    if "input_field" in spec:
        value_name = spec["value_name"]
        input_field = spec["input_field"]
        input_value = np.asarray([row[input_field] for row in selected_rows], dtype=float)
        output_value = np.asarray([row[value_name + "_real"] for row in selected_rows], dtype=float)
        slope = fitted_slope(input_value, output_value)
        residual = output_value - slope * input_value if np.isfinite(slope) else np.full_like(output_value, np.nan)

        axes[0, column].scatter(input_value, output_value, s=24)
        axes[0, column].set_title(spec["label"])
        axes[0, column].set_xlabel(input_field)
        axes[0, column].set_ylabel(value_name)
        axes[0, column].grid(alpha=0.25)

        axes[1, column].scatter(input_value, residual, s=24)
        axes[1, column].axhline(0, color="black", linestyle="--", linewidth=1)
        axes[1, column].set_xlabel(input_field)
        axes[1, column].set_ylabel("residual")
        axes[1, column].grid(alpha=0.25)
        continue

    value_name = spec["value_name"]
    amp_field = spec["amp_field"]
    phase_field = spec["phase_field"]
    order = UNO_HARMONIC_ORDERS[value_name]
    period = 360.0 / order
    input_amp = np.asarray([row[amp_field] for row in selected_rows], dtype=float)
    input_phase = np.asarray([row[phase_field] % period for row in selected_rows], dtype=float)
    output_amp = np.asarray([row[value_name + "_abs"] for row in selected_rows], dtype=float)
    output_phase = np.asarray([row[value_name + "_phase_deg"] for row in selected_rows], dtype=float)

    axes[0, column].scatter(input_amp, output_amp, c=input_phase, cmap="twilight", s=22)
    axes[0, column].set_title(spec["label"])
    axes[0, column].set_xlabel(amp_field)
    axes[0, column].set_ylabel(f"abs({value_name})")
    axes[0, column].grid(alpha=0.25)

    axes[1, column].scatter(input_phase, output_phase, c=input_amp, cmap="viridis", s=22)
    axes[1, column].plot([0, period], [0, period], color="black", linestyle="--", linewidth=1)
    axes[1, column].set_xlim(-0.04 * period, 1.04 * period)
    axes[1, column].set_ylim(-0.04 * period, 1.04 * period)
    axes[1, column].set_xlabel(f"{phase_field} mod period")
    axes[1, column].set_ylabel(f"{value_name}_phase_deg")
    axes[1, column].grid(alpha=0.25)

fig.suptitle("Uno coefficient relationships: scalar values, amplitude, and phase", fontsize=14)
fig.tight_layout()
summary_plot_path = OUTPUT_DIR / "uno_coefficient_relationship_summary.png"
fig.savefig(summary_plot_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", summary_plot_path)

## 11. Probe Shape Gallery

These grids show representative simulated probe images for each one-coefficient sweep, in the same spirit as the probe-shape panels in Fig. 8 of Uno et al. Each coefficient family is shown separately. The top row uses `C1_offset=-909 nm`; the bottom row uses `C1_offset=+909 nm`.

In [ ]:
def safe_label(label):
    return (
        label.lower()
        .replace("/", "_")
        .replace(" ", "_")
        .replace("+", "p")
        .replace("-", "m")
    )


def representative_probe_rows(spec, max_columns=6):
    selected_rows = [row for row in rows if row["sweep_label"] == spec["label"]]
    if not selected_rows:
        return []

    if "input_field" in spec:
        selected_rows = sorted(selected_rows, key=lambda row: row[spec["input_field"]])
    else:
        selected_rows = sorted(selected_rows, key=lambda row: (row[spec["amp_field"]], row[spec["phase_field"]]))

    if len(selected_rows) <= max_columns:
        return selected_rows
    indices = np.linspace(0, len(selected_rows) - 1, max_columns).round().astype(int)
    return [selected_rows[int(index)] for index in indices]


def probe_case_title(spec, row):
    if "input_field" in spec:
        return f"{spec['input_field']}={row[spec['input_field']]:g}"
    return f"{spec['amp_field']}={row[spec['amp_field']]:g}\n{spec['phase_field']}={row[spec['phase_field']]:g} deg"


def probe_edge_fraction(image, border=8):
    edge_mask = np.zeros(image.shape, dtype=bool)
    edge_mask[:border, :] = True
    edge_mask[-border:, :] = True
    edge_mask[:, :border] = True
    edge_mask[:, -border:] = True
    total = float(np.nansum(image))
    if total <= 0:
        return np.nan
    return float(np.nansum(image[edge_mask]) / total)


def probe_display_limits(image):
    vmax = float(np.nanpercentile(image, 99.8))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(image))
    return 0.0, vmax


def plot_probe_shape_gallery_for_spec(spec, max_columns=6):
    selected_rows = representative_probe_rows(spec, max_columns=max_columns)
    if not selected_rows:
        raise ValueError(f"No probe rows found for {spec['label']}.")

    fig, axes = plt.subplots(2, len(selected_rows), figsize=(2.35 * len(selected_rows), 4.7), squeeze=False)
    diagnostics = []
    for col, row in enumerate(selected_rows):
        for axis_row, index_key, offset_label in [
            (0, "under_index", f"C1_offset={UNDER_FOCUS_C1_OFFSET} nm"),
            (1, "over_index", f"C1_offset={OVER_FOCUS_C1_OFFSET} nm"),
        ]:
            image = probe_np[:, :, int(row[index_key])]
            vmin, vmax = probe_display_limits(image)
            edge_fraction = probe_edge_fraction(image)
            peak = float(np.nanmax(image))
            diagnostics.append((probe_case_title(spec, row).replace("\n", ", "), offset_label, peak, edge_fraction))

            axis = axes[axis_row, col]
            im = axis.imshow(image, cmap="magma", vmin=vmin, vmax=vmax)
            fig.colorbar(im, ax=axis, fraction=0.046, pad=0.02)
            axis.set_xticks([])
            axis.set_yticks([])
            axis.text(
                0.03,
                0.05,
                f"edge={100 * edge_fraction:.1f}%",
                transform=axis.transAxes,
                color="white",
                fontsize=7,
                bbox={"facecolor": "black", "alpha": 0.45, "pad": 1.5},
            )
            if col == 0:
                axis.set_ylabel(offset_label, fontsize=9)
            if axis_row == 0:
                axis.set_title(probe_case_title(spec, row), fontsize=9)

    fig.suptitle(f"Probe shapes: {spec['label']} | each panel scaled independently", fontsize=12)
    fig.tight_layout()
    plot_path = OUTPUT_DIR / f"probe_shapes_{safe_label(spec['label'])}.png"
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", plot_path)
    print(f"{spec['label']} probe diagnostics: peak intensity and 8-pixel edge intensity fraction")
    for title, offset_label, peak, edge_fraction in diagnostics:
        print(f"  {title} | {offset_label}: peak={peak:.4g}, edge={100 * edge_fraction:.2f}%")
    return plot_path


probe_shape_paths = [plot_probe_shape_gallery_for_spec(spec) for spec in SWEEP_SPECS]

## 12. Download Bundle

In [ ]:
REMOTE_DOWNLOAD_DIR = ROOT / "Downloads from Colab"
REMOTE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = OUTPUT_DIR / "uno_coefficient_relationships.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(CSV_PATH, CSV_PATH.name)
    zf.write(summary_plot_path, summary_plot_path.name)
    for path in plot_paths:
        zf.write(path, path.name)
    for path in probe_shape_paths:
        zf.write(path, path.name)
print("saved:", ZIP_PATH)

REMOTE_ZIP_PATH = REMOTE_DOWNLOAD_DIR / ZIP_PATH.name
REMOTE_CSV_PATH = REMOTE_DOWNLOAD_DIR / CSV_PATH.name
REMOTE_NPZ_PATH = REMOTE_DOWNLOAD_DIR / NPZ_PATH.name
for source, target in [
    (ZIP_PATH, REMOTE_ZIP_PATH),
    (CSV_PATH, REMOTE_CSV_PATH),
    (NPZ_PATH, REMOTE_NPZ_PATH),
]:
    target.write_bytes(source.read_bytes())
    print("copied inside Colab VM:", target)

print()
print("Important: the copied folder above is inside the remote Colab runtime, not your local Mac repo folder.")
print("Colab cannot directly write to /Users/.../Downloads from Colab on the local machine.")
print("The browser or VS Code controls where files.download() lands locally.")

try:
    from google.colab import files
    files.download(str(REMOTE_ZIP_PATH))
    files.download(str(REMOTE_CSV_PATH))
except Exception as exc:
    print("Browser download skipped or unsupported in this frontend:", exc)
    print("Download manually from the Colab file browser:", REMOTE_DOWNLOAD_DIR)
